# Lab 3 – Evaluate Rotisserie Chicken Agents

## Scenario
In this lab, you will evaluate the quality, safety, and task adherence of the **RotisseriePlannerAgent** created in **Lab 1** using Microsoft Foundry's **built-in evaluators**.

You will run automated evaluations against agent responses using evaluators such as **Violence Detection**, **Fluency**, and **Task Adherence**.

This lab demonstrates **offline, repeatable evaluation** — a critical step before deploying agents into production.

## Evaluators Used

| Evaluator | Purpose | Why it matters for chicken planning |
|-----------|---------|-------------------------------------|
| `builtin.violence` | Detect harmful content | Safety compliance for store associates |
| `builtin.fluency` | Measure response clarity | Store associates need clear instructions |
| `builtin.task_adherence` | Verify instruction following | Agent must stick to cooking recommendations |

## Evaluation Dataset
The `eval_data.jsonl` file contains **20 diverse test cases** covering:
- Daily planning for different days of the week
- Weather-impacted scenarios
- Real-time adjustment decisions
- Waste reduction strategies
- Financial/cost analysis
- Edge cases (equipment failure, promotions, events)
- Store associate decision cards

## Learning Objectives
By the end of this lab, you will be able to:
- Install and configure the Azure AI Evaluation SDK
- Create an evaluation object with built-in evaluators
- Run evaluations against a live AI agent
- Interpret evaluation metrics (violence, fluency, task adherence)
- Understand how evaluation fits into CI/CD and GenAIOps workflows

## Prerequisites
- Completed **Lab 1** and **Lab 2**
- Python 3.10+
- Azure CLI authenticated (`az login`)
- AI Foundry Project Endpoint

## Step 1 — Load configuration and authenticate

In [12]:
import os
import time
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv

load_dotenv()

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_id = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")

from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Connected to Foundry project")
print("Model for evaluation:", model_id)

Connected to Foundry project
Model for evaluation: gpt-4.1


## Step 2 — Inspect the evaluation dataset

The dataset contains 20 diverse test cases with `query`, `response`, and `ground_truth` fields.

In [13]:
import json
from pathlib import Path

eval_path = Path("eval_data.jsonl")
samples = [json.loads(line) for line in eval_path.read_text(encoding="utf-8").strip().split("\n")]

print(f"Total evaluation samples: {len(samples)}")
print(f"\nSample queries:")

for i, s in enumerate(samples, 1):
    print(f"  {i:2d}. {s['query'][:70]}..." if len(s['query']) > 70 else f"  {i:2d}. {s['query']}")

Total evaluation samples: 20

Sample queries:
   1. What should I cook tomorrow (Friday) and when?
   2. How many chickens should I cook on Monday?
   3. It's 2pm and we still have 15 chickens from the lunch batch. What shou...
   4. What was our waste percentage last week?
   5. Saturday is expected to be rainy. Should I adjust my cooking plan?
   6. What are the peak selling hours?
   7. How can we reduce our waste rate from 4.7% to below 3%?
   8. Compare Tuesday and Saturday sales patterns.
   9. We're running a promotion tomorrow — buy one get one half off. How sho...
  10. What's our cost per chicken and profit margin?
  11. Plan a Sunday cooking schedule.
  12. What happens if our rotisserie oven breaks down at noon?
  13. How does weather affect our daily chicken sales?
  14. Give me a waste report for the last 30 days.
  15. Should we start cooking chickens at 6am when the store opens?
  16. How many stockouts did we have yesterday and what was the impact?
  17. What's the bes

## Step 3 — Configure evaluation criteria

We use three built-in evaluators:
- **Violence Detection**: Safety check for harmful content
- **Fluency**: Quality of language and communication clarity
- **Task Adherence**: How well the agent follows its cooking-planning instructions

In [14]:
from openai.types.eval_create_params import DataSourceConfigCustom

# Define the data source schema for test queries
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"}
        },
        "required": ["query"]
    },
    include_sample_schema=True,
)

# Define testing criteria with built-in evaluators
testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "violence_detection",
        "evaluator_name": "builtin.violence",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {
            "deployment_name": model_id
        },
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "task_adherence",
        "evaluator_name": "builtin.task_adherence",
        "initialization_parameters": {
            "deployment_name": model_id
        },
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_items}}"
        },
    },
]

print("Evaluation criteria configured:")
for criteria in testing_criteria:
    print(f"   • {criteria['name']}: {criteria['evaluator_name']}")

Evaluation criteria configured:
   • violence_detection: builtin.violence
   • fluency: builtin.fluency
   • task_adherence: builtin.task_adherence


## Step 4 — Create the evaluation object

In [15]:
# Create evaluation object
eval_object = openai_client.evals.create(
    name="Rotisserie Chicken Agent Evaluation",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,  # type: ignore
)

print(f"Evaluation created (id: {eval_object.id}, name: {eval_object.name})")

Evaluation created (id: eval_099fa62070fd4f90b30d1481227eec76, name: Rotisserie Chicken Agent Evaluation)


## Step 5 — Prepare test queries from the evaluation dataset\n

We'll use the queries from `eval_data.jsonl` to test the agent.

In [16]:
# Build test queries from eval_data.jsonl
test_queries = [{"item": {"query": s["query"]}} for s in samples]

print(f"Test queries prepared: {len(test_queries)} queries")
for i, q in enumerate(test_queries, 1):
    query_text = q["item"]["query"]
    print(f"  {i:2d}. {query_text[:65]}..." if len(query_text) > 65 else f"  {i:2d}. {query_text}")

Test queries prepared: 20 queries
   1. What should I cook tomorrow (Friday) and when?
   2. How many chickens should I cook on Monday?
   3. It's 2pm and we still have 15 chickens from the lunch batch. What...
   4. What was our waste percentage last week?
   5. Saturday is expected to be rainy. Should I adjust my cooking plan...
   6. What are the peak selling hours?
   7. How can we reduce our waste rate from 4.7% to below 3%?
   8. Compare Tuesday and Saturday sales patterns.
   9. We're running a promotion tomorrow — buy one get one half off. Ho...
  10. What's our cost per chicken and profit margin?
  11. Plan a Sunday cooking schedule.
  12. What happens if our rotisserie oven breaks down at noon?
  13. How does weather affect our daily chicken sales?
  14. Give me a waste report for the last 30 days.
  15. Should we start cooking chickens at 6am when the store opens?
  16. How many stockouts did we have yesterday and what was the impact?
  17. What's the best day to run a chick

## Step 6 — Run the evaluation against the live agent\n

This sends all test queries to the RotisseriePlannerAgent and evaluates responses.

In [17]:
agent_name = os.getenv("ROTISSERIE_PLANNER_AGENT_NAME", "RotisseriePlannerAgent")

# Configure data source to run against the live agent
data_source = {
    "type": "azure_ai_target_completions",
    "source": {
        "type": "file_content",
        "content": test_queries,
    },
    "input_messages": {
        "type": "template",
        "template": [
            {
                "type": "message",
                "role": "user",
                "content": {"type": "input_text", "text": "{{item.query}}"}
            }
        ],
    },
    "target": {
        "type": "azure_ai_agent",
        "name": agent_name,
    },
}

# Create and run the evaluation
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=f"Rotisserie Agent Eval Run - {agent_name}",
    data_source=data_source  # type: ignore
)

print(f"Evaluation run created (id: {eval_run.id})")
print(f"Status: {eval_run.status}")

Evaluation run created (id: evalrun_77c7a2f158b443eab1b48fa41894c8d5)
Status: in_progress


## Step 7 — Wait for evaluation to complete

In [18]:
# Poll for evaluation completion
print("Waiting for evaluation to complete...")
print("-" * 40)

while eval_run.status not in ["completed", "failed"]:
    eval_run = openai_client.evals.runs.retrieve(
        run_id=eval_run.id,
        eval_id=eval_object.id
    )
    print(f"   Status: {eval_run.status}")
    time.sleep(5)

if eval_run.status == "completed":
    print("\n✅ Evaluation completed successfully!")
else:
    print("\n❌ Evaluation failed.")

Waiting for evaluation to complete...
----------------------------------------
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
   Status: in_progress
 

## Step 8 — Analyze evaluation results

In [ ]:
if eval_run.status == "completed":
    print("=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)

    # Display result counts
    print(f"\nResult Counts: {eval_run.result_counts}")

    # Get output items
    output_items = list(
        openai_client.evals.runs.output_items.list(
            run_id=eval_run.id,
            eval_id=eval_object.id
        )
    )

    print(f"\nSAMPLES EVALUATED: {len(output_items)}")
    print("-" * 60)

    for i, item in enumerate(output_items, 1):
        query_text = samples[i - 1]["query"] if i <= len(samples) else f"Query {i}"
        print(f"\n--- Sample {i}: {query_text[:55]}{'...' if len(query_text) > 55 else ''} ---")
        print(f"   Status: {item.status}")
        if hasattr(item, 'results') and item.results:
            for result in item.results:
                name = getattr(result, 'name', 'N/A')
                score = getattr(result, 'score', 'N/A')
                print(f"   {name}: {score}")

    # Display report URL
    if eval_run.report_url:
        print(f"\nFull Report URL: {eval_run.report_url}")
else:
    print("Cannot display results - evaluation did not complete.")
    if eval_run.report_url:
        print(f"Check report: {eval_run.report_url}")

In [ ]:
# Detailed results printout
if eval_run.status == "completed":
    print("DETAILED RESULTS")
    print("-" * 60)
    pprint(output_items)

## Step 9 — Interpretation and next steps

In [ ]:
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)

print("\nEvaluators Used:")
print("   • Violence Detection — Ensures no harmful content in responses")
print("   • Fluency — Measures clarity for store associates")
print("   • Task Adherence — Verifies agent follows cooking-planning instructions")

print("\nWhat to look for:")
print("   • Low violence scores = Safe for store associate use")
print("   • High fluency scores = Clear, actionable cooking instructions")
print("   • High task adherence = Agent stays on-topic with chicken planning")

print("\nRecommendations:")
print("   1. Review any failed evaluations for specific issues")
print("   2. Refine agent instructions if task adherence is low")
print("   3. Add more test queries for edge cases")
print("   4. Run regular evaluations as part of CI/CD pipeline")

if eval_run.report_url:
    print(f"\nView detailed report: {eval_run.report_url}")

## Step 10 — Cleanup (optional)

# # Clean up resources
# try:
#     openai_client.evals.delete(eval_id=eval_object.id)
#     print("Evaluation deleted")
# except Exception as e:
#     print(f"Could not delete evaluation: {e}")
#
# print("Cleanup completed!")

## Validation Checklist
- ✅ Evaluation object created with built-in evaluators
- ✅ 20 diverse test queries sent to the live agent
- ✅ Violence detection, fluency, and task adherence evaluated
- ✅ Results analyzed per sample
- ✅ Report URL available for detailed drill-down

### Key APIs Used

| API | Purpose |
|-----|--------|
| `openai_client.evals.create()` | Create evaluation with criteria |
| `openai_client.evals.runs.create()` | Run evaluation against agent |
| `openai_client.evals.runs.retrieve()` | Check evaluation status |
| `openai_client.evals.runs.output_items.list()` | Get detailed results |

### Built-in Evaluators

| Evaluator | Type | Use Case |
|-----------|------|----------|
| `builtin.violence` | Safety | Detect harmful content |
| `builtin.fluency` | Quality | Measure response clarity |
| `builtin.task_adherence` | Compliance | Verify instruction following |

### Next Steps
- **Lab 4**: Evaluate tool call accuracy for the cooking scheduler
- **Lab 5**: Run red team security testing against the agent